<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit3/session-11-state-and-memory/notebook.ipynb)


# Session 11 — State and memory

**Goal:** hold state on purpose — one preference and a capped episode list, a written retention policy, and a per-user store that cannot hand one person another's memory.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (llama3.2 at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. Conversation state against long-term memory

- **Conversation state**: this session only; it dies with reset.
- **Semantic memory**: facts, such as "the user prefers Portuguese docs".
- **Episodic memory**: examples of past interactions.
- **Procedural memory**: how-to patterns. Skills are procedural memory.

Every stored item needs four answers: why it is useful, who owns it, how it is corrected, when it expires. An item with no answer to the last one is a leak waiting to happen, and an item with no owner is readable by everyone (section 5).

## 2. A minimal, explicit state wrapper

The container is written for you. What it DOES with the state is the exercise.

In [4]:
from dataclasses import dataclass, field

from bootcamp_agent.agent import AgentResult, answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)


@dataclass
class SessionState:
    # ONE preference and ONE episode list. Scope creep starts at two.
    preferences: dict[str, str] = field(default_factory=dict)
    episodes: list[str] = field(default_factory=list)

    def reset(self) -> None:
        self.preferences.clear()
        self.episodes.clear()


## 3. Exercise: make the state actually do something

**Context.** Storing state is easy. Two things make it a feature: the stored preference must change what the model sees, and the store must have a ceiling. Both are one line, and both are usually missing.

**Instructions.**

1. The episode append is done. Add (a): when `preferences['answer_style'] == 'short'`, append ` (answer briefly)` to the question BEFORE it reaches the model.
2. Add (b): keep at most the last 5 episodes. Expiry is a feature, not a nicety.
3. `reset()` is already written on the dataclass above; the check exercises it.
4. Run the cell, then run the check. It sets the preference, watches the prompt the model saw, resets, then asks 7 questions and counts the episodes.

In [5]:
def answer_with_state(question: str, state: SessionState, client) -> AgentResult:
    decorated = question
    if state.preferences.get("answer_style") == "short":
        decorated = f"{question} (answer briefly)"
    # TODO(you): (a) if the answer_style preference is "short", append " (answer briefly)"
    result = answer_question(decorated, documents, client)
    state.episodes.append(f"Q: {question[:60]}")  # done
    state.episodes = state.episodes[-5:]
    return result


probe = FakeLLM()
state = SessionState()
state.preferences["answer_style"] = "short"
answer_with_state("How does chunking work in RAG?", state, probe)
print("prompt the model saw:", probe.calls[-1][1][-40:] if probe.calls else "(no call)")
print("episodes:", state.episodes)

prompt the model saw: s chunking work in RAG? (answer briefly)
episodes: ['Q: How does chunking work in RAG?']


**Expected output** (yours may differ in wording, not in shape):

```
prompt the model saw: ...work in RAG? (answer briefly)
episodes: ['Q: How does chunking work in RAG?']
✅ ch11-e1 passed
```

In [6]:
check("ch11-e1", {"state_cls": SessionState, "answer_fn": answer_with_state})

✅ ch11-e1 passed


True

## 4. Exercise: the storage policy

**Context.** Write the policy for YOUR capstone before you store anything. The last line is the one that matters, and it is the one nobody writes.

**Instructions.**

1. Fill every line: what is stored, why, how it is corrected, when it expires.
2. Then the refusal list: personal data, secrets, anything you cannot justify keeping.
3. Run the check. It refuses a policy whose labels are there and whose answers are not.

In [7]:
storage_policy = """
STORED: the answer_style preference (e.g. 'short'), and the last 5 question episodes
WHY: the preference changes what the model sees on every call, so storing it once avoids repeating the instruction every question; the episode list gives the next answer session-level context without re-reading the whole history
CORRECTED BY: the user, at any time, by stating a new preference in plain language (e.g. 'give me longer answers'); the new value overwrites the old one immediately
EXPIRES: the episode list caps at 5 entries (oldest dropped first); both the preference and episodes are cleared on reset() or when the session ends, whichever comes first
WE REFUSE TO REMEMBER: any question or answer text containing a password, API key, card number, or other credential; any health, legal or financial detail about the user; the content of a refused answer, since a refusal already means the data was too sensitive to touch once, let alone store
"""

# TODO(you): answer every line
print(storage_policy)


STORED: the answer_style preference (e.g. 'short'), and the last 5 question episodes
WHY: the preference changes what the model sees on every call, so storing it once avoids repeating the instruction every question; the episode list gives the next answer session-level context without re-reading the whole history
CORRECTED BY: the user, at any time, by stating a new preference in plain language (e.g. 'give me longer answers'); the new value overwrites the old one immediately
EXPIRES: the episode list caps at 5 entries (oldest dropped first); both the preference and episodes are cleared on reset() or when the session ends, whichever comes first
WE REFUSE TO REMEMBER: any question or answer text containing a password, API key, card number, or other credential; any health, legal or financial detail about the user; the content of a refused answer, since a refusal already means the data was too sensitive to touch once, let alone store



**Expected output** (yours may differ in wording, not in shape):

```
STORED: one answer_style preference; the last 5 question summaries (episodes).
...
✅ ch11-e2 passed
```

In [8]:
check("ch11-e2", storage_policy)

✅ ch11-e2 passed


True

## 5. Exercise: memory that belongs to one person

**Context.** The state above lives in one process, for one conversation. The moment your assistant has two users, every memory needs an owner. The store below takes a `user_id`, ignores it, and keys everything on `key` alone — the shape most first drafts have, because a one-user test cannot fail on it. Run the cell and read the first line: ana asked for her locale and got bruno's.

**Instructions.**

1. (a) Refuse a `user_id` that is empty or `None`. Raise `ValueError`, in **both** functions, before touching the store. A memory with no owner is one bucket every caller can read.
2. (b) Key the store on the owner as well as the key, so `('ana', 'locale')` and `('bruno', 'locale')` are two memories.
3. (c) Return `None` when this user never stored this key. Not a raise, not a default, and never the value somebody else has.
4. (d) Copy the value in both directions — `copy.deepcopy` going in and coming out. A reference you hand out is a second writer you never declared.
5. Run the cell, then the check. It drives your two functions with two users, a key nobody stored, a missing id, and a list it edits after reading.

In [9]:
import copy
from typing import Any


class MemoryStore:
    """Per-user memory. One process, many users, and nothing shared between them."""

    def __init__(self) -> None:
        self._values: dict[Any, Any] = {}

    def remember(self, user_id: str, key: str, value: Any) -> None:
        if not user_id:
            raise ValueError("user_id must not be empty or None")
        self._values[(user_id, key)] = copy.deepcopy(value)

    def recall(self, user_id: str, key: str) -> Any | None:
        if not user_id:
            raise ValueError("user_id must not be empty or None")
        stored = self._values.get((user_id, key))
        if stored is None:
            return None
        return copy.deepcopy(stored)


store = MemoryStore()
store.remember("ana", "locale", "pt-BR")
store.remember("bruno", "locale", "en-GB")
print("ana asked for her locale ->", store.recall("ana", "locale"))
print("bruno asked for his      ->", store.recall("bruno", "locale"))

ana asked for her locale -> pt-BR
bruno asked for his      -> en-GB


**Expected output** (yours may differ in wording, not in shape):

```
ana asked for her locale -> pt-BR
bruno asked for his      -> en-GB
✅ ch11-e3 passed
```

As shipped, both lines say `en-GB`. That is the bug, and it is the whole exercise.

In [10]:
check("ch11-e3", MemoryStore)

✅ ch11-e3 passed


True

## 6. Failure injection: memory that was true once

The store now keeps each person's memory to themselves. It still cannot say whether any of it is still true.

An entry that never recorded when it stops being true cannot be checked for staleness. It can only be quoted. Run this.

In [11]:
from datetime import date

today = date(2026, 11, 1)
memories = [
    {"key": "locale", "value": "pt-BR", "expires": date(2026, 10, 28)},
    {"key": "plan", "value": "free tier", "expires": None},
]
for item in memories:
    fresh = item["expires"] is not None and item["expires"] >= today
    verdict = "quote it" if fresh else "read it again from the source"
    print(f"{item['key']:7} {item['value']:10} expires {str(item['expires']):10} -> {verdict}")

locale  pt-BR      expires 2026-10-28 -> read it again from the source
plan    free tier  expires None       -> read it again from the source


**Expected output:**

```
locale  pt-BR      expires 2026-10-28 -> read it again from the source
plan    free tier  expires None       -> read it again from the source
```

Same verdict, two different reasons. The locale expired, and you can see that it did. The plan never said when it stopped being true, so nothing here can tell you whether it is stale — and a memory nobody can check is not fresh, it is unaudited. `EXPIRES:` in your policy is the line that makes `quote it` reachable at all.

## Exit ticket

Homework: add a 'what we refuse to remember' section to your project README.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [12]:
review("ch11")

ch11: 3/3 passed  ·  300/300 marks


True